In [1]:
from pathlib import Path
from io import StringIO
import pandas as pd
import re

In [2]:
DIR = Path("Consolidated_Schedule_of_Investments/2026-03-31/")
DIR_FILES = Path(DIR, "soi_forms")
OUTDIR = Path(DIR, "outputs")
OUTDIR.mkdir(exist_ok=True)

In [3]:
SCHEDULE_ANCHORS = [
    'id="csoi"',
    "Consolidated Schedule of Investments"
]

END_MARKERS = [
    "Consolidated Statement of Assets and Liabilities",
    '<span style="color:#000000;white-space:pre-wrap;font-weight:bold;font-size:10pt;font-family:Times New Roman;font-kerning:none;min-width:fit-content;">December 31, 2025</span>',
    "Statement of Assets and Liabilities",
    "Consolidated Statement of Operations",
    "Commitment Type",
    "Statement of Operations",
    # 'ix:footnote id="fn-1"',
    # "Financial Highlights",
]

industry_list = (
    "Aerospace and Defense",
    "Auto Components",
    "Automobiles",
    "Banks",
    "Beverages",
    "Biotechnology",
    "Building Products",
    "Capital Markets",
    "Chemicals",
    "Commercial Services and Supplies",
    "Communications Equipment",
    "Construction and Engineering",
    "Construction Materials",
    "Consumer Finance",
    "Consumer Staples Distribution and Retail",
    "Containers and Packaging",
    "Distributors",
    "Diversified Consumer Services",
    "Diversified Financial Services",
    "Diversified Telecommunication Services",
    "Electric Utilities",
    "Electrical Equipment",
    "Electronic Equipment, Instruments and Components",
    "Energy Equipment and Services",  # Consolidated & fixed missing comma after this
    "Entertainment",
    "Financial Services",
    "Food Products",
    "Gas Utilities",
    "Ground Transportation",
    "Health Care Equipment and Supplies",
    "Health Care Technology",
    "Hotels, Restaurants and Leisure",
    "Household Durables",
    "Healthcare Providers and Services", 
    "Household Products",
    "Industrial Conglomerates",
    "Insurance",
    "Interactive Media and Services",
    "Internet and Catalog Retail",
    "Internet and Direct Marketing Retail",
    "Internet Software and Services",  # Fixed missing comma after this
    "IT Services",
    "Life Sciences Tools and Services",
    "Machinery",
    "Media",
    "Metals and Mining",
    "Oil, Gas and Consumable Fuels",
    "Paper and Forest Products",
    "Pharmaceuticals",
    "Professional Services",
    "Real Estate Management and Development",
    "Road and Rail",
    "Semiconductors and Semiconductor Equipment",
    "Software",
    "Specialty Retail",
    "Trading Companies and Distributors",
    "Textiles, Apparel and Luxury Goods",
    "Technology Hardware, Storage and Peripherals",
    "Transportation Infrastructure",
    "Wireless Telecommunication Services",
    
)


instrument = (
    "First Lien Debt",
    "Second Lien Debt",
    "Unsecured Debt",
    "Structured Finance Obligations",
    "Equity and other"
)


investment_type = (
    "Debt Investments(A)",
    "Equity Securities",
    "First Lien Debt - non-controlled/non-affiliated",
    "First Lien Debt - non-controlled/affiliated",
    "First Lien Debt - controlled/affiliated",
    "Second Lien Debt - non-controlled/non-affiliated",
    "Unsecured Debt - non-controlled/non-affiliated",
    "Structured Finance Obligations - Debt Instruments - non-controlled/non-affiliated",
    "Structured Finance Obligations - Equity Instruments - non-controlled/non-affiliated",
    "Equity and other - non-controlled/non-affiliated",
    "Equity and other - non-controlled/affiliated",
    "Equity and other - controlled/affiliated (excluding Investments in Joint Ventures)",
    "Investments in Joint Ventures",
    "Cash and Cash Equivalents",
)

CURRENCIES = {"USD", "EUR", "GBP", "NOK", "CAD", "CHF", "JPY", "AUD"}
NUM_PAT = re.compile(r"^\(?[\d,]+(?:\.\d+)?\)?$")
FOOTNOTE_SUFFIX_PAT = re.compile(r"^[\)\u200b\s]*")

In [4]:
def extract_schedule_html(text: str) -> str:
    start = -1
    
    # returns the index of the first match if found, otherwise it returns -1
    for anchor in SCHEDULE_ANCHORS:
        idx = text.find(anchor)
        if idx != -1:
            # print(f"Start: {anchor} --> {idx}")
            start = idx
            break          
    if start == -1:
        return ""

    # finding ending point
    end_candidates = [] # multiple end points
    for marker in END_MARKERS:
        idx = text.find(marker, start) # Search after start position.
        if idx != -1:
            # print(f"End: {marker} --> {idx}")
            end_candidates.append(idx)
            
    # if markers found choose earliest one otherwise file length
    end = min(end_candidates) if end_candidates else len(text)
    # return the portion of that file
    return text[start:end]

In [5]:
def combine_currency_amount(vals, start_index, span: int = 3) -> str:
    if start_index is None:
        return ""

    parts = []
    for idx in range(start_index, min(start_index + span, len(vals))):
        part = clean(vals[idx])
        if not part or part == "$":
            continue
        parts.append(part)

    if not parts:
        return ""

    combined = "".join(parts)
    combined = combined.replace("$", "").replace().strip()
    combined = re.sub(r"\)([^)\d].*)$", ")", combined)

    if combined.startswith("(") and not combined.endswith(")"):
        combined = f"{combined})"

    return combined.strip()

In [6]:
def normalize_header_label(label: str) -> str:
    return re.sub(r"\s+", " ", str(label)).strip().upper()

def clean(x):
    if pd.isna(x):
        return ""

    text = (
        str(x)
        .replace("\xa0", " ")
        .replace("\u200b", "")
        .replace("�", "")
    )

    return re.sub(r"\s+", " ", text).strip()

# any number of spaces / tabs / newlines replaces them with a single space
def normalize_header_label(label: str) -> str:
    val = re.sub(r"\s+", " ", label).strip().upper()
    return val


def filing_part_name(filepath: Path) -> str:
    return filepath.stem # Returns the file name without its extension.


def pick_value(vals, index):
    if index is None or index >= len(vals):
        return ""
    return vals[index]

def is_blank_row(vals) -> bool:
    return all(v == "" for v in vals)


from itertools import groupby

def is_section_row(vals, positions) -> bool:
    issuer = pick_value(vals, positions["issuer"])
    if not issuer:
        return False

    values = [v for v in vals if v]
    if not values:
        return False
    
    return all(v == values[0] or v in ["$", "—", "�"] for v in values)


def normalize_section(text):
    return clean(text).replace("(continued)", "").strip().lower()

    
def classify_section(text: str) -> str | None:
    norm = normalize_section(text)
    
    if any(normalize_section(x) == norm for x in industry_list):
        return "industry", norm
    
    elif any(normalize_section(x) == norm for x in instrument):
        return "instrument", norm
    
    elif any(normalize_section(x) == norm for x in investment_type):
        return "investment type", norm
        
    return None, text


def combine_fair_value(vals, start_index, footnotes_index) -> str:
    amount = combine_currency_amount(vals, start_index, span=2)
    if amount.startswith("(") and not amount.endswith(")"):
        close_cell = pick_value(vals, footnotes_index)
        if close_cell.startswith(")"):
            amount = f"{amount})"
    return amount


def infer_investment_type(asset_class: str, industry: str, investment_type: str) -> str:
    if investment_type:
        return investment_type

    sections = f"{asset_class} {industry}"
    for inferred, markers in INVESTMENT_TYPE_BY_SECTION:
        if any(marker in sections for marker in markers):
            return inferred
    return "Other"


def parse_footnotes(vals, footnotes_index) -> str:
    footnotes = pick_value(vals, footnotes_index)
    footnotes = FOOTNOTE_SUFFIX_PAT.sub("", footnotes).lstrip(")").strip()
    return footnotes


def is_skip_text(text: str) -> bool:
    t = text.lower()
    return (
        not t
        or "accompanying notes" in t
        or "consolidated schedule of investments" in t
        or t.startswith("total ")
        or t == "portfolio company"
    )


def normalize_numeric(v: str):
    v = clean(v)
    
    # 1. Check if the value is fully enclosed in standard brackets
    if v.startswith("(") or v.endswith(")"):
        v = f"-{v.strip('()')}"

    if v in {"", "-", "—"}:
        return pd.NA
        
    return v



def combine_rate_spread(v:str, v2:str):
    return v + " " + v2

In [7]:
def detect_header_positions(df: pd.DataFrame):
    for ridx in range(min(5, len(df))):
        vals = [clean(v) for v in df.iloc[ridx].tolist()]
        normalized = [normalize_header_label(v) for v in vals if v]
        
        has_issuer = any(v.startswith("ISSUER") for v in normalized)
        has_instrument = any(v.startswith("INSTRUMENT") for v in normalized)
        has_ref = any("REF" in v for v in normalized)
        has_floor = any(v.startswith("FLOOR") for v in normalized)
        has_spread = any(v.startswith("SPREAD") for v in normalized)
        has_total_coupon = any("TOTAL COUPON" in v or "COUPON" in v for v in normalized)
        has_maturity = any(v.startswith("MATURITY") for v in normalized)
        has_principal = any(v.startswith("PRINCIPAL") for v in normalized)
        has_cost = any(v.startswith("COST") for v in normalized)
        has_fair_value = any(v.startswith("FAIR VALUE") or v == "FAIR VALUE" for v in normalized)
        has_net_assets = any("% OF TOTAL" in v or "NET ASSETS" in v for v in normalized)
        has_notes = any(v.startswith("NOTES") for v in normalized)

        if has_issuer and has_instrument and has_total_coupon and has_maturity and has_principal and has_cost and has_fair_value:
            label_map = {normalize_header_label(v): i for i, v in enumerate(vals) if v}
            
            def find_index(*candidates: str):
                for candidate in candidates:
                    idx = label_map.get(candidate)
                    if idx is not None:
                        return idx
                return None

            def leftmost_label_index(*labels: str):
                idxs = [
                    i
                    for i, v in enumerate(vals)
                    if normalize_header_label(v) in labels
                    or any(normalize_header_label(v).startswith(label) for label in labels)
                ]
                return min(idxs) if idxs else None


            obj = {
                "header_row": ridx,
                "issuer": find_index("ISSUER(F)"),
                "instrument": find_index("INSTRUMENT"),
                "ref": find_index("REF(B)"),
                "floor": find_index("FLOOR"),
                "spread": find_index("SPREAD"),
                "total_coupon": find_index("TOTAL COUPON"),
                "principal": find_index("PRINCIPAL"),
                "maturity": find_index("MATURITY"),
                "total_cash_investment": find_index("% OF TOTAL CASH AND INVESTMENT"), 
                "cost_start": find_index("COST"),
                "fair_value_start": find_index("FAIR VALUE"),
                "notes": leftmost_label_index("NOTES")
                
            }
            
            return obj
    return None

In [8]:
def detect_piv_header_positions(df: pd.DataFrame):
    for ridx in range(min(5, len(df))):
        vals = [clean(v) for v in df.iloc[ridx].tolist()]
        normalized = [normalize_header_label(v) for v in vals if v]
        has_security = any(v.startswith("SECURITY") for v in normalized)
        has_acquisition_date = any("FIRST ACQUISITION DATE" in v for v in normalized)
        has_cost = "COST" in normalized

        if has_security and has_acquisition_date and has_cost:
            cost_idxs = [i for i, v in enumerate(vals) if normalize_header_label(v) == "COST"]
            date_idxs = [
                i for i, v in enumerate(vals) if "FIRST ACQUISITION DATE" in normalize_header_label(v)
            ]
            return {
                "header_row": ridx,
                "portfolio_company": 0,
                "first_acquisition_date": min(date_idxs) if date_idxs else None,
                "cost_start": min(cost_idxs) if cost_idxs else None,
            }
    return None

In [9]:
def parse_table(
    df: pd.DataFrame,
    part: str,
    table_index: int,
    investment_type: str = "", 
    industry: str = "", 
    instrument: str = ""
):
    positions = detect_header_positions(df)
    if not positions:
        return [], [], asset_class, industry
        
    rows = []
    rejects = []

    for row_index in range(positions["header_row"], len(df)):
        vals = [clean(v) for v in df.iloc[row_index].tolist()]
        if is_blank_row(vals):
            continue

        issuer = pick_value(vals, positions["issuer"])
        if is_section_row(vals, positions):
            section_kind, sec = classify_section(issuer)
            if section_kind == "industry":
                industry = sec
            elif section_kind == "instrument":
                instrument = sec
            elif section_kind == "investment type":
                investment_type = sec
            continue


        joined = " | ".join([v for v in vals if v])
        upper = joined.upper()
        
        if "ISSUER" in upper and "FAIR VALUE" in upper:
            continue

        
        issuer = pick_value(vals, positions.get("issuer"))
        instrument = pick_value(vals, positions.get("instrument"))
        ref = pick_value(vals, positions.get("ref"))
        floor = pick_value(vals, positions.get("floor"))
        spread = pick_value(vals, positions.get("spread"))
        total_coupon = pick_value(vals, positions.get("total_coupon"))
        maturity = pick_value(vals, positions.get("maturity"))
        principal = pick_value(vals, positions.get("principal"))
        cost = pick_value(vals, positions.get("cost_start"))
        fair_value = pick_value(vals, positions.get("fair_value_start"))
        total_cash_investment = pick_value(vals, positions.get("total_cash_investment"))
        notes = pick_value(vals, positions.get("notes"))

  
        
        cost_ok = bool(NUM_PAT.match(cost)) or cost in {"", "-", "—"}
        fair_ok = bool(NUM_PAT.match(fair_value)) or fair_value in {"", "-", "—"}

        
        if (
            issuer
            and not is_skip_text(issuer)
            and cost_ok
            and fair_ok
        ):
            
            obj = { 
                "issuer": issuer, 
                "instrument": instrument, 
                "industry": industry, 
                "ref": ref, 
                "floor": floor, 
                "investment_type": investment_type, 
                "spread": spread, 
                "total_coupon": total_coupon, 
                "maturity": maturity, 
                "principal": normalize_numeric(principal), 
                "cost": normalize_numeric(cost),
                "fair_value": normalize_numeric(fair_value),
                "total_cash_investment": normalize_numeric(total_cash_investment), 
                "notes": notes, 
                "part": part, 
                "table_index": table_index, 
                "row_index": row_index 
            }

            if pd.isna(obj["cost"]) or obj["cost"] == "":
                obj["cost"] = 0
                
            if pd.isna(obj["fair_value"]) or obj["fair_value"] == "":
                obj["fair_value"] = 0


                    
            rows.append(obj)
            
        else:
            if joined and not is_skip_text(issuer):
                rejects.append({
                    "issuer": issuer,
                    "part": part,
                    "table_index": table_index,
                    "row_index": row_index,
                    "raw_row": joined,
                })
    
    target_issuers = {"Other Cash and Cash Equivalents", "Cash and Cash Equivalents - 16.7% of Net Assets"}
    rows = [o for o in rows if o.get("issuer") not in target_issuers]

    return rows, rejects, investment_type, industry, instrument

In [10]:
def parse_piv_table(
    df: pd.DataFrame,
    part: str,
    table_index: int,
    asset_class: str = "",
    industry: str = "",
):
    positions = detect_piv_header_positions(df)
    if not positions:
        return [], [], asset_class, industry

    rows = []
    rejects = []
    for row_index in range(positions["header_row"], len(df)):
        vals = [clean(v) for v in df.iloc[row_index].tolist()]
        if is_blank_row(vals):
            continue

        portfolio_company = pick_value(vals, positions["portfolio_company"])
        if is_section_row(vals, positions):
            section_kind = classify_section(portfolio_company)
            if section_kind == "asset_class":
                asset_class = portfolio_company
            elif section_kind == "industry":
                industry = portfolio_company
            continue

        acquisition_date = pick_value(vals, positions["first_acquisition_date"])
        cost = combine_currency_amount(vals, positions["cost_start"])
        if not cost:
            numeric_cells = [v for v in vals if v and NUM_PAT.match(v)]
            cost = numeric_cells[-1] if numeric_cells else ""

        joined = " | ".join([v for v in vals if v])
        cost_ok = bool(NUM_PAT.match(cost))

        if portfolio_company and not is_skip_text(portfolio_company) and cost_ok:
            rows.append({
                "portfolio_company": portfolio_company,
                "investment_type": "Private Investment Vehicle",
                "interest_rate": "",
                "reference_rate": "",
                "basis_points_spread": "",
                "maturity_date": "",
                "currency": "",
                "principal_amount": normalize_numeric(cost),
                "cost": normalize_numeric(cost),
                "fair_value": pd.NA,
                "footnotes": "",
                "first_acquisition_date": acquisition_date,
                "asset_class": asset_class,
                "industry": industry,
                "part": part,
                "table_index": table_index,
                "row_index": row_index,
            })
            
        elif joined and not is_skip_text(portfolio_company):
            rejects.append({
                "part": part,
                "table_index": table_index,
                "row_index": row_index,
                "raw_row": joined,
            })

    return rows, rejects, asset_class, industry

In [11]:
def main():
    all_clean = []

    for filepath in sorted(DIR_FILES.iterdir()):
        if filepath.suffix.lower() not in {".htm", ".html"}:
            continue
        filename = str(filepath).split("/")[-1].split(".")[0]
        part = filing_part_name(filepath)
        html = filepath.read_text(errors="ignore")
        snippet = extract_schedule_html(html)
        tables = pd.read_html(StringIO(snippet), flavor="lxml")
        
        part_rows = []
        candidate_tables = 0
        investment_type = industry = instrument = ""
        
        # print(tables[0][24])
        
        for table_index, df in enumerate(tables):
            # if table_index == 1: break
            if detect_header_positions(df):
                candidate_tables += 1
                rows, rejects, investment_type, industry, instrument = parse_table(
                    df, part, table_index, investment_type, industry, instrument
                )
                part_rows.extend(rows)
                
            elif detect_piv_header_positions(df):
                candidate_tables += 1
                rows, rejects, asset_class, industry = parse_piv_table(
                    df, part, table_index, asset_class, industry
                )
                part_rows.extend(rows)

        part_df = pd.DataFrame(part_rows)
        if not part_df.empty:
            part_df = part_df.sort_values(["table_index", "row_index"], kind="stable").reset_index(drop=True)

        all_clean.append(part_df)
        master_df = pd.concat(all_clean, ignore_index=True) if all_clean else pd.DataFrame()

        dedup_cols = [
            "issuer",
            "instrument",
            "ref",
            "investment_type",
            "industry",
            "floor",
            "spread",
            "total_coupon",
            "maturity",
            "principal",
            "cost",
            "fair_value",
            "total_cash_investment",
            "notes"
        ]

    # print(master_df)
    if not master_df.empty:
        master_dedup_df = master_df.drop_duplicates(subset=dedup_cols, keep="first").reset_index(drop=True)
    else:
        master_dedup_df = master_df
    master_dedup_df.to_csv(OUTDIR / f"{filename}.csv", index=False)

main()